In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Bangladesh National Election 2026 – Analytics Pipeline

**Course:** CSE488 | **Date:** 2026-02-27

This notebook implements the full big-data analytics pipeline using **Apache Spark (PySpark)** and **Spark MLlib**.

### Pipeline Stages
1. **Spark Setup** – Install PySpark, initialise SparkSession
2. **Data Ingestion** – Load election, BBS, and HDX datasets into Spark DataFrames
3. **Data Engineering** – Cleaning, joins, feature engineering (Spark SQL)
4. **MLlib Analytics** – Correlation, Regression, Classification, Clustering
5. **Export** – Persist ML artifacts for dashboard consumption

---
## Stage 1 — Spark Environment Setup

In [3]:
# ── 1.1  Install PySpark (Colab already has Java pre-installed) ────────────────
!pip install -q pyspark==3.5.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 1.4 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.2 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.


In [4]:
# ── 1.2  Initialise SparkSession ──────────────────────────────────────────────
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("BD-Election-2026-Analytics")
    .master("local[*]")                          # use all available Colab cores
    .config("spark.driver.memory", "8g")         # Colab standard runtime ≈ 12 GB RAM
    .config("spark.sql.shuffle.partitions", "8") # keep small; single-node
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")           # suppress INFO noise

print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print(f"Master        : {spark.sparkContext.master}")
print(f"Cores         : {spark.sparkContext.defaultParallelism}")
spark

Spark version : 3.5.0
App name      : BD-Election-2026-Analytics
Master        : local[*]
Cores         : 2


In [ ]:
import os
os.listdir('/content/drive/MyDrive/Election-Data/') # Uploaded the local "data" directory to drive

['data']

---
## Stage 2 — Data Ingestion & Engineering

### 2A – Load raw datasets into Spark DataFrames
- **Election results**: per-candidate rows with constituency, party, votes, district, division
- **Socio-economic (HDX/BBS)**: district-level census indicators (445 columns → select key features)

In [ ]:
# ── 2A.1  Load Election Results ────────────────────────────────────────────────
DATA_ROOT = "/content/drive/MyDrive/Election-Data/data"

election_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{DATA_ROOT}/raw/news_scrape/daily_star_candidates_2026-02-20.csv"))

print(f"Election rows : {election_raw.count()}")
print(f"Columns       : {len(election_raw.columns)}")
election_raw.printSchema()
election_raw.show(5, truncate=30)

In [ ]:
# ── 2A.2  Load Socio-Economic (HDX/BBS Census) Data ───────────────────────────
# The HDX dataset has 445 columns — select only the ones relevant for ML features.

SOCIO_COLS_KEEP = [
    # Keys
    "Division", "District", "District_Geocode",
    # Demographics
    "Population_Total", "Household_Total", "`Population Density`",
    # Urban / Rural population
    "`Population by Sex, Dist & Loca_Population_Total_#`",
    "`Population by Sex, Dist & Loca_Population_rural_Total_#`",
    "`Population by Sex, Dist & Loca_Population_Urban_Total_#`",
    # Literacy
    "`Literacy Rate_7year+_Overall`", "`Literacy Rate_7year+_Male`", "`Literacy Rate_7year+_Female`",
    # Internet penetration
    "`%_Inernet_Total_5year+`", "`%_Inernet_Male_5year+`", "`%_Inernet_Female_5year+`",
    # Mobile phone usage
    "`%_Mobile Phone_Total_5year+`",
    # Employment
    "`Overall_Employed_Working_Status_5 Year+_#`",
    "`Overall_Looking for work_Working_Status_5 Year+_#`",
    # Youth NEET
    "`15-24 Years NEET_%_Overall`",
    # Financial inclusion
    "`%_Have financial account_Overall`",
    # Religion (majority share)
    "`Population by Religion, Sex_# Total_Muslim`",
    "`Population by Religion, Sex_# Total_Hindu`",
]

hdx_file = (f"{DATA_ROOT}/raw/HDX/"
            "bangladesh_bbs_population-and-housing-census-dataset_2022_admin-02"
            " - Merged_All_Table.csv")

socio_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(hdx_file))

# Select only needed columns
socio_raw = socio_raw.selectExpr(*SOCIO_COLS_KEEP)

print(f"Socio-econ rows : {socio_raw.count()}")
print(f"Columns kept    : {len(socio_raw.columns)}")
socio_raw.printSchema()
socio_raw.show(5, truncate=25)

In [ ]:
# ── 2A.3  Register Spark SQL Temp Views ────────────────────────────────────────
election_raw.createOrReplaceTempView("election_raw")
socio_raw.createOrReplaceTempView("socio_raw")

# Quick SQL sanity check
print("── Election: top 5 parties by seat count ──")
spark.sql("""
    SELECT party, COUNT(*) AS winners
    FROM election_raw
    WHERE is_winner = 'true'
    GROUP BY party
    ORDER BY winners DESC
    LIMIT 10
""").show(truncate=30)

print("── Socio-econ: sample districts ──")
spark.sql("""
    SELECT District, Population_Total, `Population Density`,
           `Literacy Rate_7year+_Overall` AS literacy_rate
    FROM socio_raw
    LIMIT 5
""").show(truncate=25)

### 2B – Data Cleaning & Standardisation

- Cast types, handle nulls, deduplicate
- Standardise district names across datasets (3 known mismatches)
- Derive winner/runner-up per constituency

In [ ]:
# ── 2B.1  Clean Election Data ──────────────────────────────────────────────────
from pyspark.sql import functions as F, Window

# 1) Cast votes to integer, is_winner to boolean
election = (election_raw
    .withColumn("votes", F.col("votes").cast("int"))
    .withColumn("is_winner", F.col("is_winner") == "true")
    .drop("page_title", "scraped_at", "source", "url")  # drop metadata cols
)

# 2) Standardise district names to match HDX/BBS spelling
DISTRICT_MAP = {
    "Coxs bazar":       "Cox's Bazar",
    "Chapainawabganj":  "Chapainababganj",
    "Jhalokathi":       "Jhalokati",
}

for old, new in DISTRICT_MAP.items():
    election = election.withColumn(
        "district",
        F.when(F.col("district") == old, new).otherwise(F.col("district"))
    )

# 3) Drop duplicates (same candidate + constituency)
before = election.count()
election = election.dropDuplicates(["constituency", "candidate_name"])
after = election.count()
print(f"Deduplication: {before} → {after} rows (removed {before - after})")

# 4) Derive per-constituency winner & runner-up using window functions
w = Window.partitionBy("constituency").orderBy(F.col("votes").desc())
election = election.withColumn("vote_rank", F.row_number().over(w))

election.createOrReplaceTempView("election")
election.filter(F.col("vote_rank") <= 2).show(10, truncate=25)

In [ ]:
# ── 2B.2  Clean & Rename Socio-Economic Data ──────────────────────────────────
# Rename long column names to short, ML-friendly aliases
socio = (socio_raw
    .withColumnRenamed("Population Density", "pop_density")
    .withColumnRenamed("Population by Sex, Dist & Loca_Population_Total_#", "pop_total_loc")
    .withColumnRenamed("Population by Sex, Dist & Loca_Population_rural_Total_#", "pop_rural")
    .withColumnRenamed("Population by Sex, Dist & Loca_Population_Urban_Total_#", "pop_urban")
    .withColumnRenamed("Literacy Rate_7year+_Overall", "literacy_rate")
    .withColumnRenamed("Literacy Rate_7year+_Male", "literacy_male")
    .withColumnRenamed("Literacy Rate_7year+_Female", "literacy_female")
    .withColumnRenamed("%_Inernet_Total_5year+", "internet_pct")
    .withColumnRenamed("%_Inernet_Male_5year+", "internet_male_pct")
    .withColumnRenamed("%_Inernet_Female_5year+", "internet_female_pct")
    .withColumnRenamed("%_Mobile Phone_Total_5year+", "mobile_phone_pct")
    .withColumnRenamed("Overall_Employed_Working_Status_5 Year+_#", "employed_total")
    .withColumnRenamed("Overall_Looking for work_Working_Status_5 Year+_#", "looking_for_work")
    .withColumnRenamed("15-24 Years NEET_%_Overall", "neet_pct")
    .withColumnRenamed("%_Have financial account_Overall", "financial_account_pct")
    .withColumnRenamed("Population by Religion, Sex_# Total_Muslim", "pop_muslim")
    .withColumnRenamed("Population by Religion, Sex_# Total_Hindu", "pop_hindu")
)

# Cast all numeric columns (inferSchema may miss some due to commas/spaces)
numeric_cols = [c for c in socio.columns if c not in ("Division", "District", "District_Geocode")]
for c in numeric_cols:
    socio = socio.withColumn(c, F.col(c).cast("double"))

# Null check per column
print("── Null counts per column ──")
socio.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in socio.columns]).show(vertical=True)

socio.createOrReplaceTempView("socio")
socio.show(5, truncate=20)

In [ ]:
# ── 2B.3  District Name Reconciliation Check ──────────────────────────────────
# Verify all election districts match a socio-economic district after cleaning.

election_districts = (spark.sql("SELECT DISTINCT district FROM election")
                      .rdd.flatMap(lambda x: x).collect())
socio_districts    = (spark.sql("SELECT DISTINCT District FROM socio")
                      .rdd.flatMap(lambda x: x).collect())

unmatched_election = set(election_districts) - set(socio_districts)
unmatched_socio    = set(socio_districts) - set(election_districts)

if unmatched_election:
    print(f"⚠  Election districts NOT in socio-econ: {unmatched_election}")
else:
    print("✓ All election districts found in socio-economic data")

if unmatched_socio:
    print(f"ℹ  Socio-econ districts with no election data: {unmatched_socio}")

print(f"\nElection districts: {len(election_districts)} | Socio-econ districts: {len(socio_districts)}")

### 2C – Joining Datasets & Feature Engineering (Spark SQL + DataFrame API)

- Build **constituency-level** summary from candidate-level election data
- Join with **district-level** socio-economic indicators
- Engineer ML features: `turnout_pct`, `winning_margin_pct`, `urbanization_index`, `competitiveness_index`

In [ ]:
# ── 2C.1  Build Constituency-Level Summary (Spark SQL) ────────────────────────
# Aggregate candidate-level data to one row per constituency using Spark SQL.

constituency = spark.sql("""
    WITH ranked AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY constituency ORDER BY votes DESC) AS rnk,
               SUM(votes)   OVER (PARTITION BY constituency) AS total_votes
        FROM election
    )
    SELECT
        constituency,
        district,
        division,
        alliance,

        -- Winner info (rank 1)
        MAX(CASE WHEN rnk = 1 THEN candidate_name END) AS winner_candidate,
        MAX(CASE WHEN rnk = 1 THEN party END)           AS winner_party,
        MAX(CASE WHEN rnk = 1 THEN votes END)           AS winner_votes,

        -- Runner-up info (rank 2)
        MAX(CASE WHEN rnk = 2 THEN candidate_name END) AS runner_up_candidate,
        MAX(CASE WHEN rnk = 2 THEN party END)           AS runner_up_party,
        MAX(CASE WHEN rnk = 2 THEN votes END)           AS runner_up_votes,

        -- Aggregates
        MAX(total_votes)    AS total_valid_votes,
        MAX(candidate_count) AS candidate_count

    FROM ranked
    GROUP BY constituency, district, division, alliance
    ORDER BY constituency
""")

constituency.createOrReplaceTempView("constituency")
print(f"Constituencies: {constituency.count()}")
constituency.show(5, truncate=22)

In [ ]:
# ── 2C.2  Join Election ↔ Socio-Economic (Spark SQL + DataFrame API) ──────────
# Join on district — constituency-level election data inherits district-level
# socio-economic indicators.

# --- Method 1: Spark SQL join ---
joined_sql = spark.sql("""
    SELECT c.*, s.*
    FROM constituency c
    LEFT JOIN socio s ON c.district = s.District
""")

# Drop duplicate join key columns
joined = (joined_sql
    .drop("Division")           # already have division from election
    .drop("District")           # duplicate of c.district
    .drop("District_Geocode")   # not needed for ML
)

joined.createOrReplaceTempView("joined")
print(f"Joined rows: {joined.count()} | Columns: {len(joined.columns)}")

# Check for any constituencies that failed to join
orphans = joined.filter(F.col("literacy_rate").isNull()).select("constituency", "district")
if orphans.count() > 0:
    print("⚠  Constituencies without socio-economic match:")
    orphans.show()
else:
    print("✓ All constituencies matched socio-economic data")

joined.show(3, truncate=18)

In [ ]:
# ── 2C.3  Feature Engineering ──────────────────────────────────────────────────
# Create derived features required by the analytics plan.

analytics_base = (joined
    # -- Winning margin % --
    .withColumn("winning_margin_pct",
        F.round((F.col("winner_votes") - F.col("runner_up_votes"))
                / F.col("total_valid_votes") * 100, 2))

    # -- Winner vote share % --
    .withColumn("winner_vote_share_pct",
        F.round(F.col("winner_votes") / F.col("total_valid_votes") * 100, 2))

    # -- Urbanization index (urban pop / total pop) --
    .withColumn("urbanization_index",
        F.round(F.col("pop_urban") / F.col("Population_Total") * 100, 2))

    # -- Employment rate % --
    .withColumn("employment_rate_pct",
        F.round(F.col("employed_total") / F.col("Population_Total") * 100, 2))

    # -- Competitiveness index (inverse margin × candidate count) --
    # Higher = more competitive
    .withColumn("competitiveness_index",
        F.round(
            F.col("candidate_count") /
            (1 + (F.col("winner_votes") - F.col("runner_up_votes"))
                 / F.col("total_valid_votes") * 100),
            2))

    # -- Muslim majority flag --
    .withColumn("muslim_majority_pct",
        F.round(F.col("pop_muslim") / F.col("Population_Total") * 100, 2))
)

analytics_base.createOrReplaceTempView("analytics_base")
print(f"analytics_base: {analytics_base.count()} rows × {len(analytics_base.columns)} cols")
analytics_base.select(
    "constituency", "winner_party", "winning_margin_pct",
    "winner_vote_share_pct", "urbanization_index",
    "literacy_rate", "internet_pct", "competitiveness_index"
).show(10, truncate=22)

In [ ]:
# ── 2C.4  Save analytics_base to Parquet & CSV ───────────────────────────────
OUTPUT_DIR = f"{DATA_ROOT}/processed/spark_output"

analytics_base.coalesce(1).write.mode("overwrite").parquet(f"{OUTPUT_DIR}/analytics_base.parquet")
analytics_base.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{OUTPUT_DIR}/analytics_base_csv")

print(f"✓ Saved analytics_base to {OUTPUT_DIR}/")
print(f"  Parquet: analytics_base.parquet")
print(f"  CSV:     analytics_base_csv/")

### 2D – Validation & Summary Statistics

Final quality checks on `analytics_base` before moving to MLlib analytics.

In [ ]:
# ── 2D.1  Quality Checks ──────────────────────────────────────────────────────
print("═" * 60)
print("ANALYTICS BASE – QUALITY REPORT")
print("═" * 60)

n = analytics_base.count()
print(f"\n1) Row count: {n}")

# Null rate for key columns
key_cols = [
    "constituency", "winner_party", "winner_votes", "total_valid_votes",
    "winning_margin_pct", "literacy_rate", "internet_pct",
    "urbanization_index", "pop_density", "neet_pct"
]
print("\n2) Null rates (key columns):")
for c in key_cols:
    nulls = analytics_base.filter(F.col(c).isNull()).count()
    print(f"   {c:30s}  {nulls:3d}/{n}  ({nulls/n*100:.1f}%)")

# Range checks for percentages (should be 0-100)
pct_cols = ["winning_margin_pct", "winner_vote_share_pct", "urbanization_index",
            "literacy_rate", "internet_pct", "neet_pct"]
print("\n3) Range checks (min / max for % columns):")
for c in pct_cols:
    stats = analytics_base.agg(F.min(c).alias("min"), F.max(c).alias("max")).first()
    flag = "" if (stats["min"] is not None and stats["min"] >= 0 and stats["max"] <= 100) else " ⚠"
    print(f"   {c:30s}  [{stats['min']}, {stats['max']}]{flag}")

# Duplicate constituency check
dup_count = n - analytics_base.select("constituency").distinct().count()
print(f"\n4) Duplicate constituencies: {dup_count}")

# Party distribution
print("\n5) Seats per party:")
spark.sql("""
    SELECT winner_party, COUNT(*) AS seats,
           ROUND(AVG(winning_margin_pct), 2) AS avg_margin,
           ROUND(AVG(literacy_rate), 2) AS avg_literacy
    FROM analytics_base
    GROUP BY winner_party
    ORDER BY seats DESC
""").show(15, truncate=30)

print("═" * 60)
print("✓ Phase 2 complete – analytics_base ready for MLlib (Phase 3)")
print("═" * 60)